# ExtraTrees 5-fold x 5-repeats for all metabolites

This notebook runs ExtraTrees on all metabolite targets and exports true vs predicted values.
Predictions are out-of-fold predictions from RepeatedKFold(5,5), plus mean prediction across 5 repeats.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import r2_score

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, *args, **kwargs):
        return x


In [ ]:
xlsx_files = [f for f in Path('.').glob('*.xlsx') if not f.name.startswith('~$')]
if not xlsx_files:
    raise FileNotFoundError('No xlsx file found in current directory.')

input_file = None
for f in xlsx_files:
    if 'annotatio.xlsx' in f.name:
        input_file = f
        break
if input_file is None:
    input_file = xlsx_files[0]

df = pd.read_excel(input_file)
print('Input file:', input_file)
print('Shape:', df.shape)

if 'ILA_CDM' not in df.columns or 'IAA_KBUFFER' not in df.columns:
    raise ValueError('Missing required boundary columns: ILA_CDM to IAA_KBUFFER')

metabolite_cols = list(df.loc[:, 'ILA_CDM':'IAA_KBUFFER'].columns)
gene_cols = list(df.columns[df.columns.get_loc('IAA_KBUFFER') + 1:])
print('Metabolites:', metabolite_cols)
print('Num gene features:', len(gene_cols))

X_all = df[gene_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
X_all = (X_all > 0).astype(np.int8)


In [ ]:
n_splits = 5
n_repeats = 5
random_state = 42

id_cols = [c for c in ['strain', 'Abbreviation', 'Type strain'] if c in df.columns]
all_pred_tables = {}
summary_rows = []

for met in tqdm(metabolite_cols, desc='Metabolites', unit='met'):
    y_raw = pd.to_numeric(df[met], errors='coerce')
    mask = y_raw.notna()

    X = X_all.loc[mask].to_numpy(dtype=np.float32)
    y = y_raw.loc[mask].to_numpy(dtype=np.float64)
    idx = y_raw.loc[mask].index.to_numpy()

    rkf = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)
    pred_by_repeat = np.full((len(y), n_repeats), np.nan, dtype=np.float64)

    for split_i, (tr_idx, te_idx) in enumerate(rkf.split(X), start=1):
        rep = (split_i - 1) // n_splits

        model = ExtraTreesRegressor(
            n_estimators=160,
            max_features='sqrt',
            min_samples_leaf=1,
            random_state=random_state,
            n_jobs=-1,
        )
        model.fit(X[tr_idx], y[tr_idx])
        pred = model.predict(X[te_idx])
        pred_by_repeat[te_idx, rep] = pred

    y_pred_mean_5repeat = np.nanmean(pred_by_repeat, axis=1)

    r2_mean_pred = r2_score(y, y_pred_mean_5repeat)
    rho_mean_pred, p_mean_pred = spearmanr(y, y_pred_mean_5repeat)

    repeat_r2, repeat_rho, repeat_p = [], [], []
    for r in range(n_repeats):
        pr = pred_by_repeat[:, r]
        repeat_r2.append(r2_score(y, pr))
        rr, pp = spearmanr(y, pr)
        repeat_rho.append(rr)
        repeat_p.append(pp)

    out = df.loc[idx, id_cols].copy() if id_cols else pd.DataFrame(index=idx)
    out.insert(len(out.columns), 'metabolite', met)
    out.insert(len(out.columns), 'true_value', y)
    for r in range(n_repeats):
        out.insert(len(out.columns), f'pred_repeat_{r+1}', pred_by_repeat[:, r])
    out.insert(len(out.columns), 'pred_mean_5repeat', y_pred_mean_5repeat)
    out.insert(len(out.columns), 'n_pred_used', np.sum(~np.isnan(pred_by_repeat), axis=1))

    all_pred_tables[met] = out

    summary_rows.append({
        'metabolite': met,
        'n_samples': len(y),
        'n_features': X.shape[1],
        'r2_mean_of_5repeat_predictions': r2_mean_pred,
        'spearman_rho_mean_of_5repeat_predictions': rho_mean_pred,
        'spearman_p_mean_of_5repeat_predictions': p_mean_pred,
        'r2_mean_across_5repeats': float(np.mean(repeat_r2)),
        'r2_std_across_5repeats': float(np.std(repeat_r2, ddof=1)),
        'spearman_rho_mean_across_5repeats': float(np.mean(repeat_rho)),
        'spearman_rho_std_across_5repeats': float(np.std(repeat_rho, ddof=1)),
        'spearman_p_median_across_5repeats': float(np.median(repeat_p)),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('r2_mean_of_5repeat_predictions', ascending=False)
summary_df

In [ ]:
out_xlsx = Path('4_ExtraTrees_5x5CV_predictions.xlsx')
out_summary_csv = Path('4_ExtraTrees_5x5CV_summary.csv')
out_long_csv = Path('4_ExtraTrees_5x5CV_long.csv')

with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='SUMMARY', index=False)
    for met, tab in all_pred_tables.items():
        tab.to_excel(writer, sheet_name=met[:31], index=False)

summary_df.to_csv(out_summary_csv, index=False, encoding='utf-8-sig')
long_df = pd.concat(all_pred_tables.values(), ignore_index=True)
long_df.to_csv(out_long_csv, index=False, encoding='utf-8-sig')

print('Done.')
print('Excel :', out_xlsx.resolve())
print('CSV   :', out_summary_csv.resolve())
print('CSV   :', out_long_csv.resolve())